# 📊 Code 1a: Fetch Raw Data from Yahoo Finance (Updated)

## Purpose
Download stock price and volume data for all NSE stocks from Yahoo Finance.

## Input
- `unique_tickers.csv` - List of all tickers

## Output
- **`data_raw_all.csv`** - Complete raw dataset with ALL tickers
- `metadata_raw_all.csv` - Metadata about each ticker
- `tickers_not_found.csv` - Tickers not available on Yahoo Finance

## Date Range
**Start Date:** 01-Jan-2011  
**End Date:** Today

## Columns in Output:
- Date, Ticker
- Open, High, Low, Close (split-adjusted)
- Volume

## Key Update:
**No Adj_Close** - We use only Close (split-adjusted) for all calculations including returns.
This is simpler and sufficient for ATR-based trading strategy.

---

**⏱️ Estimated Time:** 30-60 minutes for ~2,000 stocks

## Step 1: Install and Import Libraries

In [1]:
# Install required packages
!pip install yfinance --quiet
!pip install pandas --quiet
!pip install tqdm --quiet

print("✅ Packages installed successfully!")

✅ Packages installed successfully!


In [2]:
# Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"📅 Script run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
📅 Script run date: 2026-06-15 16:47:53


## Step 2: Configuration

In [3]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Input/Output files
TICKER_FILE = 'unique_tickers.csv'
OUTPUT_FILE = 'data_raw_all.csv'
METADATA_FILE = 'metadata_raw_all.csv'
NOT_FOUND_FILE = 'tickers_not_found.csv'

# Date range - As per your requirements
START_DATE = '2007-01-01'  # 01-Jan-2007
END_DATE = datetime.now().strftime('%Y-%m-%d')  # Today

# Download settings
BATCH_SIZE = 50
DELAY_BETWEEN_BATCHES = 2
MAX_RETRIES = 2

print("="*80)
print("CODE 1a: FETCH RAW DATA CONFIGURATION (UPDATED)")
print("="*80)
print(f"Input File: {TICKER_FILE}")
print(f"Output File: {OUTPUT_FILE}")
print(f"Date Range: {START_DATE} to {END_DATE}")
print(f"Batch Size: {BATCH_SIZE} stocks per batch")
print(f"\n💡 KEY CHANGE: Using auto_adjust=True (split-adjusted Close only)")
print(f"   No Adj_Close column - simpler and clearer methodology")
print("="*80)

CODE 1a: FETCH RAW DATA CONFIGURATION (UPDATED)
Input File: unique_tickers.csv
Output File: data_raw_all.csv
Date Range: 2007-01-01 to 2026-06-15
Batch Size: 50 stocks per batch

💡 KEY CHANGE: Using auto_adjust=True (split-adjusted Close only)
   No Adj_Close column - simpler and clearer methodology


## Step 3: Load Ticker List

**⚠️ Make sure `unique_tickers.csv` is uploaded to Colab before running this cell.**

In [4]:
# Load ticker list
print("Loading ticker list...")
print("-"*80)

df_tickers = pd.read_csv(TICKER_FILE)

# Get ticker column (handle different column names)
if 'symbol' in df_tickers.columns:
    ticker_list = df_tickers['symbol'].tolist()
elif 'Symbol' in df_tickers.columns:
    ticker_list = df_tickers['Symbol'].tolist()
else:
    ticker_list = df_tickers.iloc[:, 0].tolist()

# Clean ticker list
ticker_list = [str(ticker).strip() for ticker in ticker_list if pd.notna(ticker)]

print(f"✅ Total tickers loaded: {len(ticker_list)}")
print(f"\nFirst 10 tickers: {ticker_list[:10]}")
print(f"Last 10 tickers: {ticker_list[-10:]}")
print("-"*80)

Loading ticker list...
--------------------------------------------------------------------------------
✅ Total tickers loaded: 2535

First 10 tickers: ['20MICRONS', '21STCENMGM', '360ONE', '3IINFOLTD', '3MINDIA', '3PLAND', '5PAISA', '63MOONS', 'A2ZINFRA', 'AADHARHFC']
Last 10 tickers: ['ZENTEC', 'ZFCVINDIA', 'ZIMLAB', 'ZODIAC', 'ZODIACLOTH', 'ZOTA', 'ZUARI', 'ZUARIIND', 'ZYDUSLIFE', 'ZYDUSWELL']
--------------------------------------------------------------------------------


## Step 4: Process Tickers - Add NSE Suffix

In [5]:
# Process tickers and add .NS suffix
print("Processing tickers...")
print("-"*80)

processed_tickers = []
ticker_mapping = {}  # Maps Yahoo ticker to original ticker
excluded_tickers = []

for ticker in ticker_list:
    # Skip index names (contain spaces)
    if ' ' in ticker or ticker.startswith('NIFTY '):
        excluded_tickers.append(ticker)
        continue

    # Add .NS suffix if not present
    if not ticker.endswith('.NS') and not ticker.endswith('.BO'):
        yahoo_ticker = ticker + '.NS'
    else:
        yahoo_ticker = ticker

    processed_tickers.append(yahoo_ticker)
    ticker_mapping[yahoo_ticker] = ticker

print(f"✅ Stock tickers to download: {len(processed_tickers)}")
print(f"📋 Excluded (indices/ETFs): {len(excluded_tickers)}")
print(f"\nFirst 10 processed: {processed_tickers[:10]}")
print("-"*80)

Processing tickers...
--------------------------------------------------------------------------------
✅ Stock tickers to download: 2415
📋 Excluded (indices/ETFs): 120

First 10 processed: ['20MICRONS.NS', '21STCENMGM.NS', '360ONE.NS', '3IINFOLTD.NS', '3MINDIA.NS', '3PLAND.NS', '5PAISA.NS', '63MOONS.NS', 'A2ZINFRA.NS', 'AADHARHFC.NS']
--------------------------------------------------------------------------------


## Step 5: Download Data from Yahoo Finance

**⏱️ This will take 30-60 minutes. Please be patient!**

**☕ Perfect time for a coffee break!**

### What's Being Downloaded:
- Date
- Open, High, Low, Close (split-adjusted)
- Volume

**Note:** Using `auto_adjust=True` which provides split-adjusted prices only.

In [6]:
print("="*80)
print("DOWNLOADING DATA FROM YAHOO FINANCE")
print("="*80)
print(f"Total tickers: {len(processed_tickers)}")
print(f"Estimated time: {len(processed_tickers) * 2 / 60:.1f} minutes")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"\n💡 Using auto_adjust=True (split-adjusted Close only)")
print("\n⏳ Starting download... Please wait.\n")

# Initialize storage
all_data = []
not_found_tickers = []
metadata_list = []

# Calculate batches
total_batches = (len(processed_tickers) + BATCH_SIZE - 1) // BATCH_SIZE

# Download in batches
for batch_num in range(total_batches):
    start_idx = batch_num * BATCH_SIZE
    end_idx = min((batch_num + 1) * BATCH_SIZE, len(processed_tickers))
    batch_tickers = processed_tickers[start_idx:end_idx]

    print(f"\n{'='*80}")
    print(f"BATCH {batch_num + 1}/{total_batches} - Tickers {start_idx + 1} to {end_idx}")
    print(f"{'='*80}")

    # Download each ticker
    for ticker in tqdm(batch_tickers, desc=f"Batch {batch_num + 1}"):
        original_ticker = ticker_mapping[ticker]
        retry_count = 0
        success = False

        while retry_count <= MAX_RETRIES and not success:
            try:
                # Download with auto_adjust=True to get split-adjusted prices
                # This gives us: Open, High, Low, Close, Volume
                # All prices are split-adjusted automatically
                data = yf.download(
                    ticker,
                    start=START_DATE,
                    end=END_DATE,
                    auto_adjust=True,  # 🔑 KEY CHANGE: Get split-adjusted prices
                    progress=False
                )

                if not data.empty and len(data) > 0:
                    # Successfully downloaded
                    data = data.reset_index()

                    # Rename columns (auto_adjust=True gives: Date, Open, High, Low, Close, Volume)
                    data.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

                    # Add ticker
                    data['Ticker'] = original_ticker

                    # Store data
                    all_data.append(data)

                    # Store metadata
                    metadata_list.append({
                        'Ticker': original_ticker,
                        'Yahoo_Ticker': ticker,
                        'First_Date': data['Date'].min(),
                        'Last_Date': data['Date'].max(),
                        'Total_Days': len(data),
                        'Avg_Volume': data['Volume'].mean(),
                        'Status': 'Success'
                    })

                    success = True

                else:
                    if retry_count == MAX_RETRIES:
                        not_found_tickers.append({
                            'Ticker': original_ticker,
                            'Yahoo_Ticker': ticker,
                            'Reason': 'No data available'
                        })
                    retry_count += 1

            except Exception as e:
                if retry_count == MAX_RETRIES:
                    error_msg = str(e)
                    not_found_tickers.append({
                        'Ticker': original_ticker,
                        'Yahoo_Ticker': ticker,
                        'Reason': 'Ticker not found' if 'No data found' in error_msg else error_msg[:100]
                    })
                retry_count += 1
                time.sleep(0.5)

    # Wait between batches
    if batch_num < total_batches - 1:
        print(f"\n⏳ Waiting {DELAY_BETWEEN_BATCHES} seconds before next batch...")
        time.sleep(DELAY_BETWEEN_BATCHES)

print("\n" + "="*80)
print("✅ DOWNLOAD COMPLETED!")
print("="*80)
print(f"Successfully downloaded: {len(all_data)} tickers")
print(f"Not found: {len(not_found_tickers)} tickers")

DOWNLOADING DATA FROM YAHOO FINANCE
Total tickers: 2415
Estimated time: 80.5 minutes
Date range: 2007-01-01 to 2026-06-15

💡 Using auto_adjust=True (split-adjusted Close only)

⏳ Starting download... Please wait.


BATCH 1/49 - Tickers 1 to 50


Batch 1: 100%|██████████| 50/50 [00:15<00:00,  3.17it/s]



⏳ Waiting 2 seconds before next batch...

BATCH 2/49 - Tickers 51 to 100


Batch 2: 100%|██████████| 50/50 [00:15<00:00,  3.30it/s]



⏳ Waiting 2 seconds before next batch...

BATCH 3/49 - Tickers 101 to 150


...


⏳ Waiting 2 seconds before next batch...

BATCH 48/49 - Tickers 2351 to 2400


Batch 48: 100%|██████████| 50/50 [00:16<00:00,  3.01it/s]



⏳ Waiting 2 seconds before next batch...

BATCH 49/49 - Tickers 2401 to 2415


Batch 49: 100%|██████████| 15/15 [00:05<00:00,  2.74it/s]


✅ DOWNLOAD COMPLETED!
Successfully downloaded: 2399 tickers
Not found: 16 tickers


## Step 6: Combine Data

In [7]:
if len(all_data) > 0:
    print("Combining all downloaded data...")
    print("-"*80)

    # Combine all dataframes
    df_raw_all = pd.concat(all_data, ignore_index=True)

    # Sort by Ticker and Date
    df_raw_all = df_raw_all.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    print(f"✅ Combined data shape: {df_raw_all.shape}")
    print(f"   Total records: {len(df_raw_all):,}")
    print(f"   Unique stocks: {df_raw_all['Ticker'].nunique()}")
    print(f"   Date range: {df_raw_all['Date'].min()} to {df_raw_all['Date'].max()}")
    print(f"   Columns: {list(df_raw_all.columns)}")
    print(f"\n💡 Note: All prices are split-adjusted. Volume is actual historical volume.")

else:
    print("❌ No data was downloaded successfully!")
    df_raw_all = pd.DataFrame()

Combining all downloaded data...
--------------------------------------------------------------------------------
✅ Combined data shape: (6435047, 7)
   Total records: 6,435,047
   Unique stocks: 2399
   Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']

💡 Note: All prices are split-adjusted. Volume is actual historical volume.


## Step 7: Save Output Files

**Note:** We're saving pure raw data with no calculations. Value_Traded will be calculated in Code 1b.

In [8]:
print("="*80)
print("SAVING OUTPUT FILES")
print("="*80)

# Save main data file
if not df_raw_all.empty:
    print(f"\nSaving main dataset to {OUTPUT_FILE}...")
    df_raw_all.to_csv(OUTPUT_FILE, index=False)
    file_size_mb = df_raw_all.memory_usage(deep=True).sum() / 1024**2
    print(f"✅ Saved: {OUTPUT_FILE}")
    print(f"   Size: {file_size_mb:.2f} MB")
    print(f"   Rows: {len(df_raw_all):,}")
    print(f"   Columns: {len(df_raw_all.columns)} - {list(df_raw_all.columns)}")

# Save metadata
if len(metadata_list) > 0:
    df_metadata = pd.DataFrame(metadata_list)
    df_metadata.to_csv(METADATA_FILE, index=False)
    print(f"\n✅ Saved: {METADATA_FILE}")
    print(f"   Tickers: {len(df_metadata)}")

# Save not found tickers
if len(not_found_tickers) > 0:
    df_not_found = pd.DataFrame(not_found_tickers)
    df_not_found.to_csv(NOT_FOUND_FILE, index=False)
    print(f"\n📋 Saved: {NOT_FOUND_FILE}")
    print(f"   Tickers not found: {len(not_found_tickers)}")

print("\n" + "="*80)
print("✅ ALL FILES SAVED!")
print("="*80)

SAVING OUTPUT FILES

Saving main dataset to data_raw_all.csv...
✅ Saved: data_raw_all.csv
   Size: 640.79 MB
   Rows: 6,435,047
   Columns: 7 - ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']

✅ Saved: metadata_raw_all.csv
   Tickers: 2399

📋 Saved: tickers_not_found.csv
   Tickers not found: 16

✅ ALL FILES SAVED!


## Step 9: Display Summary Statistics

In [9]:
print("="*80)
print("FINAL SUMMARY - CODE 1a (UPDATED)")
print("="*80)

print(f"""
📊 Download Statistics:
   - Total tickers attempted: {len(processed_tickers)}
   - Successfully downloaded: {len(all_data)}
   - Not found: {len(not_found_tickers)}
   - Success rate: {len(all_data)/len(processed_tickers)*100:.1f}%

📈 Output Dataset (data_raw_all.csv):
   - Total records: {len(df_raw_all):,}
   - Unique stocks: {df_raw_all['Ticker'].nunique() if not df_raw_all.empty else 0}
   - Date range: {df_raw_all['Date'].min() if not df_raw_all.empty else 'N/A'} to {df_raw_all['Date'].max() if not df_raw_all.empty else 'N/A'}
   - Columns: {list(df_raw_all.columns) if not df_raw_all.empty else []}

💡 PURE DATA FETCH:
   - No calculations or derived columns
   - Raw data: Date, Ticker, Open, High, Low, Close, Volume
   - All prices are split-adjusted (auto_adjust=True)
   - Value_Traded will be calculated in Code 1b

📁 Output Files:
   1. {OUTPUT_FILE} - Pure raw dataset (no calculations)
   2. {METADATA_FILE} - Metadata
   3. {NOT_FOUND_FILE} - Tickers not found

➡️  Next Step: Run Code 1b to calculate metrics and filter liquid securities
""")

print("="*80)

FINAL SUMMARY - CODE 1a (UPDATED)

📊 Download Statistics:
   - Total tickers attempted: 2415
   - Successfully downloaded: 2399
   - Not found: 16
   - Success rate: 99.3%

📈 Output Dataset (data_raw_all.csv):
   - Total records: 6,435,047
   - Unique stocks: 2399
   - Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   - Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']

💡 PURE DATA FETCH:
   - No calculations or derived columns
   - Raw data: Date, Ticker, Open, High, Low, Close, Volume
   - All prices are split-adjusted (auto_adjust=True)
   - Value_Traded will be calculated in Code 1b

📁 Output Files:
   1. data_raw_all.csv - Pure raw dataset (no calculations)
   2. metadata_raw_all.csv - Metadata
   3. tickers_not_found.csv - Tickers not found

➡️  Next Step: Run Code 1b to calculate metrics and filter liquid securities



## Step 10: Preview Data

In [10]:
if not df_raw_all.empty:
    print("SAMPLE DATA (First 20 rows):")
    print("="*80)
    display(df_raw_all.head(20))

    print("\nSUMMARY STATISTICS:")
    print("="*80)
    display(df_raw_all.describe())

SAMPLE DATA (First 20 rows):


,Date,Open,High,Low,Close,Volume,Ticker
0,2008-10-06,14.586985,34.782677,13.739157,34.782677,23501600,20MICRONS
1,2008-10-07,13.065244,16.521773,12.108721,13.913072,9113400,20MICRONS
2,2008-10-08,11.521763,12.695679,10.913066,12.173938,2464384,20MICRONS
3,2008-10-10,10.086977,10.826108,9.413062,10.826108,1207928,20MICRONS
4,2008-10-13,10.717412,11.565240,10.130454,10.565238,898692,20MICRONS
5,2008-10-14,9.630454,11.304371,9.347845,11.152196,673088,20MICRONS
6,2008-10-15,10.586977,11.108717,8.717408,9.260887,1126836,20MICRONS
7,2008-10-16,11.521763,12.717417,9.391324,9.586976,1944000,20MICRONS
8,2008-10-17,9.543497,12.478285,9.543497,12.195675,1806200,20MICRONS
9,2008-10-20,9.456540,10.108715,8.565234,9.782628,722818,20MICRONS



SUMMARY STATISTICS:


,Date,Open,High,Low,Close,Volume
count,6435047,6.435047e+06,6.435047e+06,6.435047e+06,6.435047e+06,6.435047e+06
mean,2018-07-08 12:46:40.917165568,4.630929e+02,4.715324e+02,4.559743e+02,4.639223e+02,1.554164e+06
min,2007-01-02 00:00:00,2.176401e-02,2.253396e-02,2.090782e-02,0.000000e+00,0.000000e+00
25%,2014-01-21 00:00:00,3.005413e+01,3.085000e+01,2.942426e+01,3.014229e+01,1.137800e+04
50%,2019-05-16 00:00:00,9.570000e+01,9.805609e+01,9.396782e+01,9.597636e+01,7.230800e+04
75%,2023-05-17 00:00:00,3.153717e+02,3.225000e+02,3.097189e+02,3.160000e+02,4.388410e+05
max,2026-06-12 00:00:00,1.622886e+05,1.635935e+05,1.602787e+05,1.619736e+05,2.350717e+09
std,NaN,2.410892e+03,2.442562e+03,2.383349e+03,2.414682e+03,1.152079e+07


## 📥 Download Files (Optional)

Run this cell to download files to your computer.

In [11]:
from google.colab import files

print("Downloading files...")
files.download(OUTPUT_FILE)
files.download(METADATA_FILE)
if len(not_found_tickers) > 0:
    files.download(NOT_FOUND_FILE)

print("\n✅ Download complete!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!


---

## ✅ Code 1a Complete! (Updated - Pure Fetch Only)

**Next Step:** Run **Code 1b** to calculate metrics and filter liquid securities.

### What This Code Does:
- ✅ Pure data fetch from Yahoo Finance
- ✅ Using `auto_adjust=True` for split-adjusted prices
- ✅ No Adj_Close column
- ✅ No calculations - just raw data
- ✅ Output: Date, Ticker, Open, High, Low, Close, Volume

### What Code 1b Will Do:
- ✅ Calculate Value_Traded = Close × Volume
- ✅ Filter by liquidity
- ✅ Filter by trading days
- ✅ Filter by data completeness